In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

Note: Tensorflow might still print warnings despite working correctly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
import tensorflow as tf

from func_file_Model import ResNet_model_paper

## Vizualize aberrated PSFs

In [ ]:
#Load the aberrated PSFs
datafile = np.load("Data_files/Aberrated_PSFs/All_stacks_psf_532nm.npz", allow_pickle=True)
all_psf_stacks = datafile["psf_all_stacks"]

print("Shape of all_psf_stacks:", all_psf_stacks.shape)

In [ ]:
#Visualize the upsampled aberrated PSFs
plt.figure(figsize=(15,10))
plt.subplot(231)
plt.matshow(all_psf_stacks[0, 0], fignum=False)
plt.title("Non-aberrated PSF")

plt.subplot(232)
plt.matshow(all_psf_stacks[0, -1], fignum=False)
plt.title("Coma")

plt.subplot(233)
plt.matshow(all_psf_stacks[1, -1], fignum=False)
plt.title("Astigmatism")

plt.subplot(234)
plt.matshow(all_psf_stacks[2, -1], fignum=False)
plt.title("Spherical aberration")

plt.subplot(235)
plt.matshow(all_psf_stacks[3, -1], fignum=False)
plt.title("Defocus")

plt.subplot(236)
plt.matshow(all_psf_stacks[4, 0], fignum=False)
plt.title("Low numerical aperture")

plt.show()

In [ ]:
#Vizualize the aberrated PSFs in the low-resolution image dimensions
def Remap_to_smaller_grid(image):
    remapped = np.sum(image[4:-4,4:-4].reshape([49,8,49,8]), axis=(-3,-1))
    return remapped
    
plt.figure(figsize=(15,10))
plt.subplot(231)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[0, 0]), fignum=False)
plt.title("Non-aberrated PSF")

plt.subplot(232)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[0, -1]), fignum=False)
plt.title("Coma")

plt.subplot(233)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[1, -1]), fignum=False)
plt.title("Astigmatism")

plt.subplot(234)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[2, -1]), fignum=False)
plt.title("Spherical aberration")

plt.subplot(235)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[3, -1]), fignum=False)
plt.title("Defocus")

plt.subplot(236)
plt.matshow(Remap_to_smaller_grid(all_psf_stacks[4, 0]), fignum=False)
plt.title("Low numerical aperture")

plt.show()

## Evaluate the generalization ability of the DAMN model

### Load model

Note: During loading, tensorflow warning may be printed

In [ ]:
#Load the model
model_ResNet = ResNet_model_paper(channels = 64, num_blocks_array = [3,3,3,3], 
                                  kernel_sizes=[5,7,9,11], LeakyReLU_slope=0.1, 
                                  padding="same", interpolation="bilinear", kernel_initializer="he_uniform")

#Compile and load weights
_ = model_ResNet(tf.random.normal([1, 50, 50, 1]))
model_ResNet.load_weights("DAMN_upsampling_model.keras")

### Load and normalize data

In [ ]:
#Subtract minimum and norm to unit sum
def Normalize_data(data):
    #Subtract minima
    data_subbed = data - np.expand_dims(data.min(axis=(-1,-2)), axis=(-1,-2))
    
    #Normalize to unit sum
    norms = data_subbed.sum(axis=(-1,-2))
    data_in = data_subbed / np.expand_dims(norms, axis=(-1,-2))
    
    #Normalize images of other sizes than the default 50x50 
    data_in_renormed = data_in * (data.shape[-2]*data.shape[-1]) / (50*50)
    
    return data_in_renormed

In [ ]:
# Load dataset
datafile = np.load("Data_files/Low_resolution_images/Generated_data_with_abberations_532nm.npz", allow_pickle=True)

all_stacks = Normalize_data(datafile["all_stacks"])
ground_truth = Normalize_data(datafile["ground_truth"])
aberration_dict = datafile["aberrations_dictionary"].item()

print("Low-resolution data shape:", all_stacks.shape)
print("Ground-truth data shape:  ", all_stacks.shape)

### Apply the model

The first run can be slower than the subsequent as the model is initialized.

In [ ]:
#First run to initialize everything. Takes approximatelly 1 min on our computational resources.

model_ResNet.predict(np.expand_dims(all_stacks[0,0,:], axis=-1), verbose=0)
print("Done initializing.")

In [ ]:
# Reconstruct all data using the model. Takes approximately 10 min on our computational resources.
evaluation_batch = 32   #Set the evaluation batch_size depending on your available GPU memory

predicted = np.reshape(model_ResNet.predict(all_stacks.reshape([all_stacks.shape[0] * all_stacks.shape[1] * all_stacks.shape[2], all_stacks.shape[-2], all_stacks.shape[-1], 1]), 
                                            batch_size=evaluation_batch, verbose=1), 
                       [all_stacks.shape[0], all_stacks.shape[1], all_stacks.shape[2], ground_truth.shape[-2], ground_truth.shape[-1]])
predicted.shape

### Evaluate the metric

In [ ]:
#Evaluating the metric used for the upsampling model, as per the paper Methods
def Convolved_MAE(image_1, image_2):
    convolved_image_1 = gaussian_filter(image_1, 2)
    convolved_image_2 = gaussian_filter(image_2, 2)
    
    metric = np.mean(np.abs(convolved_image_1 - convolved_image_2))
    return metric

evaluated_MAE = np.zeros([predicted.shape[0], predicted.shape[1], predicted.shape[2]])

for i in range(predicted.shape[0]):
    for j in range(predicted.shape[1]):
        for k in range(predicted.shape[2]):
            evaluated_MAE[i,j,k] = Convolved_MAE(predicted[i,j,k], ground_truth[k])

evaluated_MAE_mean = np.mean(evaluated_MAE, axis=-1)
evaluated_MAE_std = np.std(evaluated_MAE, axis=-1)

print("Evaluated MAE shape:", evaluated_MAE.shape)

In [ ]:
#And rescale the metric relative to the non-aberrated baseline
rescaled_MAE = np.zeros(evaluated_MAE.shape)
rescaled_MAE[:4] = evaluated_MAE[:4,:,:] / evaluated_MAE[:4,0,:][:,None,:]    #For each aberration type: each image of the 100 stack with strength-zero [0] serves as a baseline
rescaled_MAE[4] = evaluated_MAE[4,:,:] / evaluated_MAE[4,-1,:][None,:]        #For NA: each image of the 100 stack with maximal-NA [-1] serves as a baseline

rescaled_MAE_mean = np.mean(rescaled_MAE, axis=-1)
rescaled_MAE_std = np.std(rescaled_MAE, axis=-1)

### Visualize metric results

In [ ]:
#Colors and labels
labels = {'A_coma_x' :  "Coma", 
          'A_ast' : "Astigmatism", 
          'A_sph' : "Spherical", 
          'A_def' : "Defocus"}

colors = {'A_coma_x' :  "red", 
          'A_ast' : "blue", 
          'A_sph' : "green", 
          'A_def' : "purple"}

In [ ]:
#Visualize
plt.figure(figsize=(15,5))

#Aberrations
plt.subplot(121)
for i in range(4):
    plt.plot(aberration_dict[[key for key in aberration_dict.keys()][i]], 
             rescaled_MAE_mean[i], 
             color = colors[[key for key in aberration_dict.keys()][i]], 
             label = labels[[key for key in aberration_dict.keys()][i]])
    plt.fill_between(aberration_dict[[key for key in aberration_dict.keys()][i]], 
                     rescaled_MAE_mean[i] + rescaled_MAE_std[i], 
                     rescaled_MAE_mean[i] - rescaled_MAE_std[i], 
                     color = colors[[key for key in aberration_dict.keys()][i]], alpha=0.1)
    
plt.xlabel("Aberration strength [$\lambda$]")
plt.ylabel("Relative increase of MAE")
plt.ticklabel_format(axis="y", style="sci", scilimits=(0,0))
plt.ylim(top=1.055, bottom=0.995)
plt.grid()
plt.legend()

#Numerical aperture
plt.subplot(122)
plt.plot(aberration_dict[[key for key in aberration_dict.keys()][-1]], 
         rescaled_MAE_mean[-1], color="black")
plt.fill_between(aberration_dict[[key for key in aberration_dict.keys()][-1]], 
                 rescaled_MAE_mean[-1] + rescaled_MAE_std[-1], 
                 rescaled_MAE_mean[-1] - rescaled_MAE_std[-1], 
                 color="black", alpha=0.1)
plt.xlabel("Numerical aperture")
plt.ylabel("Relative increase of MAE")
plt.ticklabel_format(axis="y", style="sci", scilimits=(0,0))
plt.ylim(top=1.055, bottom=0.995)
plt.grid()

plt.show()